# StockSense: learn by running

A journey from random sampling to independently evaluated decisions. Run from this repository with Python 3.11-3.13 after installing it. All results are synthetic. Do not describe them as actual business impact.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd()/"stocklab").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
import numpy as np
from dataclasses import replace
from stocklab.model import Scenario, uniforms, worlds, outcomes, analytic_order, analytic_expected_profit
from stocklab.experiment import run
from stocklab.calibration import calibrate

## 1. Randomness with a replay button
A seed lets a teammate repeat the same experiment. Repeating the same sequence does not create an independent experiment.

In [ ]:
rng = np.random.default_rng(2026)
flips = rng.integers(0, 2, 10000)
print("Estimated heads probability:", flips.mean())

## 2. Monte Carlo integration with a known answer
Estimate pi by sampling points in a unit square. The fraction inside a quarter-circle estimates pi/4. This teaches simulation, but the inventory application adds a meaningful business decision.

In [ ]:
points = np.random.default_rng(42).random((100000,2))
pi_estimate = 4*np.mean(np.sum(points**2,axis=1) <= 1)
print("Estimate:", pi_estimate, "Absolute error:", abs(pi_estimate-np.pi))

## 3. Check the accounting by hand
For order=100 and demand=80, the expected result is 1,360. For demand=120 it is 2,100. Verify the separate revenue, salvage, cost, and shortage components before trusting a large simulation.

In [ ]:
s = Scenario()
profit, sold, unmet = outcomes(s, 100, np.array([80.,120.]), np.ones(2))
np.testing.assert_array_equal(profit, [1360,2100])
print(profit)

## 4. Inspect the demand model
CV is relative spread, not log-space sigma. Check that the generated moments approximately match mean=1000 and CV=.35.

In [ ]:
demand, fill = worlds(s, uniforms(16384,"sobol",8))
print({"mean":float(demand.mean()),"cv":float(demand.std()/demand.mean()),"disruption_rate":float(np.mean(fill<1))})

## 5. Run the full decision experiment
The policy is selected on one sample and evaluated on separate streams. Sobol points within a replicate are not independent; the confidence interval uses independent replicate means.

In [ ]:
result = run(s, n=4096, replicates=12, seed=2026)
print("Selected:",result["selected"])
print("Paired difference CI:",result["paired_delta_ci95"])

## 6. Compare preferences
Changing the weight changes the objective. Lower expected profit can be an intentional trade-off for improved lower-tail outcomes.

In [ ]:
for weight in [0,.2,.5]:
    r=run(replace(s,risk_weight=weight),n=1024,replicates=8)
    print(weight, r["selected"]["order"], round(r["selected"]["mean_profit"],2), round(r["selected"]["tail_profit"],2))

## 7. Verify against independent mathematics
Remove disruption and tail preference. The classical newsvendor model gives a closed-form optimum and expected profit. The full disrupted model does not share this simple oracle.

In [ ]:
simple = replace(s, disruption_probability=0, risk_weight=0)
q = analytic_order(simple)
truth = analytic_expected_profit(simple,q)
d,f = worlds(simple,uniforms(65536,"sobol",17))
estimate = outcomes(simple,q,d,f)[0].mean()
print({"optimal_order":q,"analytic_mean":truth,"simulated_mean":estimate,"absolute_error":abs(estimate-truth)})
assert abs(estimate-truth)<2

## 8. Fit synthetic demand data
Calibration is parameter estimation, not validation. Real sales may be censored by stockouts and may contain seasonal structure that this simple fit ignores.

In [ ]:
fitted = calibrate(root/"examples/synthetic-demand.csv")
print(fitted)

## 9. Read the sampling benchmark
At equal sample counts compare RMSE against the analytic truth. Do not turn one low-dimensional result into a universal accuracy or speed claim.

In [ ]:
import json
benchmark=json.loads((root/"examples/benchmark.json").read_text())
for row in benchmark["rows"]:
    print(row["method"],row["n"],"RMSE",round(row["rmse"],4))

## 10. Your extension
Change the contract to refund undelivered units. Add a test whose expected profit you calculate independently. Then explain which change came from economics and which came from simulation noise. See docs/STUDY_GUIDE.md for the answer key and advanced extensions.